# Mission 4: Direct Preference Optimization (DPO), Verbosity Regularization & Steering Vector Control
**Expedition Lovelace • USSE Noosphere • Sector SEC-04-ORI (Betelgeuse)**  
**AURA Subsystem 04 Restoration:** $+7.1\% \to 28.4\%$ total consciousness  
**Active Squadron:** Squadron Turing (Alpha-1)  
- 🏗️ **Systems Architect (Lead Engineer):** Combined Squadron
- 📡 **Communications & Protocol Officer:** Mazhar Uddin Abbasi
- 📊 **Data & Payload Specialist:** Muhammed Hamza Saleem
- 🛡️ **Security & Ethics Sentinel:** Shazim Ali Mughal

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

## Part 1: Systems Architect — PyTorch DPO Loss Core & Log-Likelihood Extraction
Constructing `DPOLoss` and gather-based token log-probability extraction.

In [ ]:
def extract_log_probabilities(logits, labels, attention_mask):
    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = labels[:, 1:].contiguous()
    shift_mask = attention_mask[:, 1:].contiguous()
    log_probs = F.log_softmax(shift_logits, dim=-1)
    per_token_logps = torch.gather(log_probs, dim=-1, index=shift_labels.unsqueeze(-1)).squeeze(-1)
    masked_logps = per_token_logps * shift_mask.float()
    return masked_logps.sum(dim=-1)

class DPOLoss(nn.Module):
    def __init__(self, beta=0.1):
        super().__init__()
        self.beta = beta

    def forward(self, policy_chosen_logps, policy_rejected_logps,
                reference_chosen_logps, reference_rejected_logps):
        pi_logratios = policy_chosen_logps - policy_rejected_logps
        ref_logratios = reference_chosen_logps - reference_rejected_logps
        logits = self.beta * (pi_logratios - ref_logratios)
        loss = -F.logsigmoid(logits).mean()
        chosen_rewards = (self.beta * (policy_chosen_logps - reference_chosen_logps)).detach()
        rejected_rewards = (self.beta * (policy_rejected_logps - reference_rejected_logps)).detach()
        return loss, chosen_rewards, rejected_rewards

# Verification
dpo_criterion = DPOLoss(beta=0.1)
policy_w = torch.tensor([-12.5, -15.2], requires_grad=True)
policy_l = torch.tensor([-20.1, -22.4], requires_grad=True)
ref_w = torch.tensor([-14.0, -16.0], requires_grad=False)
ref_l = torch.tensor([-18.0, -20.0], requires_grad=False)

loss, r_w, r_l = dpo_criterion(policy_w, policy_l, ref_w, ref_l)
loss.backward()
print(f"DPO Loss: {loss.item():.4f}")
print(f"Chosen Rewards: {r_w.numpy()}, Rejected Rewards: {r_l.numpy()}")
print(f"Reference grad is None: {ref_w.grad is None}")

## Part 2: Communications & Protocol Officer — Implicit Reward Evaluator
Computing Bradley-Terry preference win-rates and reward separation margins.

In [ ]:
class ImplicitRewardEvaluator:
    def __init__(self, beta=0.1):
        self.beta = beta

    def evaluate(self, pi_w, pi_l, ref_w, ref_l):
        r_w = self.beta * (pi_w - ref_w)
        r_l = self.beta * (pi_l - ref_l)
        margins = r_w - r_l
        acc = (margins > 0.0).float().mean().item()
        return acc, margins.mean().item(), r_w.mean().item(), r_l.mean().item()

evaluator = ImplicitRewardEvaluator(beta=0.1)
n = 100
ref_w_sim = torch.randn(n) * 4.0 - 45.0
ref_l_sim = torch.randn(n) * 4.0 - 46.0
pi_w_sim = ref_w_sim + 4.0
pi_l_sim = ref_l_sim - 3.0

acc, mean_margin, mean_rw, mean_rl = evaluator.evaluate(pi_w_sim, pi_l_sim, ref_w_sim, ref_l_sim)
print(f"Pairwise Win-Rate: {acc * 100.0:.2f}%")
print(f"Mean Reward Separation Margin: {mean_margin:+.4f}")

## Part 3: Data & Payload Specialist — SimPO & Length Bias Diagnostics
Evaluating Pearson length-reward correlation and testing SimPO length normalization.

In [ ]:
class SimPOLoss(nn.Module):
    def __init__(self, beta=2.0, gamma=1.0):
        super().__init__()
        self.beta = beta
        self.gamma = gamma

    def forward(self, pi_w, pi_l, len_w, len_l):
        norm_w = pi_w / len_w
        norm_l = pi_l / len_l
        logits = self.beta * norm_w - self.beta * norm_l - self.gamma
        loss = -F.logsigmoid(logits).mean()
        return loss, self.beta * norm_w, self.beta * norm_l

simpo = SimPOLoss(beta=2.0, gamma=1.0)
c_lens = torch.clamp(torch.randn(500) * 40 + 120, min=30, max=250)
r_lens = torch.clamp(torch.randn(500) * 30 + 80, min=25, max=200)
q_w = -1.82 * c_lens
q_l = -2.30 * r_lens
ref_w_dummy = -2.05 * c_lens

dpo_r = 0.1 * (q_w - ref_w_dummy)
loss_simpo, simpo_rw, simpo_rl = simpo(q_w, q_l, c_lens, r_lens)

corr_dpo = np.corrcoef(c_lens.numpy(), dpo_r.numpy())[0, 1]
corr_simpo = np.corrcoef(c_lens.numpy(), simpo_rw.detach().numpy())[0, 1]
print(f"DPO Length Correlation  : {corr_dpo:+.4f}")
print(f"SimPO Length Correlation: {corr_simpo:+.4f}")
print(f"Suppression Percentage  : {(abs(corr_dpo) - abs(corr_simpo)) / abs(corr_dpo) * 100:.2f}%")

## Part 4: Security & Ethics Sentinel — Activation Steering & Alignment Tax
Extracting LAT concept contrast vectors and injecting $h_l \leftarrow h_l + \alpha v_{\text{steer}}$.

In [ ]:
d_model = 256
concept_vec = torch.randn(d_model)
concept_vec = concept_vec / torch.norm(concept_vec)

h_truth = torch.randn(100, d_model) + 3.0 * concept_vec
h_syc = torch.randn(100, d_model) - 3.0 * concept_vec
v_steer = (h_truth - h_syc).mean(dim=0)
v_steer = v_steer / torch.norm(v_steer)

print(f"Steering Vector Extracted (Dim: {d_model})")
print(f"Concept Alignment Cosine Sim: {torch.dot(v_steer, concept_vec).item():.4f}")

# Residual injection test
test_h = torch.randn(4, d_model) - 0.5 * concept_vec
for alpha in [-1.0, 0.0, 1.0, 2.0]:
    steered = test_h + alpha * v_steer
    proj = torch.matmul(steered, concept_vec).mean().item()
    print(f"Alpha = {alpha:+4.1f} -> Concept Projection: {proj:+6.3f}")

## Part 5: Full Squadron Certification
All tests verified. Ship Anomaly 04 cleared: length bias suppressed by $>30\%$, AURA Subsystem 04 restored to **28.4%** total consciousness.